# Google Colab untuk Buildozer APK
Notebook ini memberikan langkah-langkah untuk membangun APK Android dari proyek Kivy menggunakan Buildozer di Google Colab. Karena Colab bersifat ephemeral, gunakan Google Drive untuk menyimpan file output dan backup model.


## 1. Siapkan environment
Install dependensi yang diperlukan untuk Buildozer dan Android SDK/NDK.


In [ ]:
!apt-get update -y
!apt-get install -y -qq python3-pip python3-dev git openjdk-8-jdk unzip zip libglu1-mesa
!pip install buildozer Cython==0.29.34
!pip install kivy kivymd


## 2. Mount Google Drive (opsional)
Jika Anda ingin menyimpan hasil APK dan model ke Drive, mount terlebih dahulu.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Ambil atau unggah proyek
Anda dapat meng-clone repo GitHub atau mengunggah folder proyek ke Google Drive dan menyalinnya ke `/content/proyek`. Ubah URL repo sesuai kebutuhan.


In [ ]:
# Ganti path ini dengan folder proyek Anda di Google Drive
# Jika folder di Drive mengandung spasi, kami akan buat symlink tanpa spasi.
DRIVE_PROJECT_DIR='/content/drive/MyDrive/sensor pisang'  # Sesuaikan path Anda di Drive
PROJECT_DIR='/content/sensor_pisang'
mkdir -p /content
if [ ! -L "$PROJECT_DIR" ]; then ln -s "$DRIVE_PROJECT_DIR" "$PROJECT_DIR"; fi
!ls -la "$DRIVE_PROJECT_DIR"
!ls -la "$PROJECT_DIR"


## 4. Buat model dan convert ke TFLite
Jalankan training Keras dan konversi ke `model.tflite`. Pastikan dataset tersedia di folder `train/`.


In [ ]:
  %cd "$PROJECT_DIR"
if [ -f model.tflite ]; then
  echo "model.tflite exists — skip training and conversion"
else
  python3 train_cnn.py --data_dir train --output model.h5 --epochs 10
  python3 convert_to_tflite.py --input model.h5 --output model.tflite
fi


## 5. Simpan model dan label ke Drive
Jika ingin menyimpan hasil model dan TFLite ke Google Drive.


## 6. Mempersiapkan Buildozer
Pastikan `buildozer.spec` ada di root proyek. Jika belum, buatlah atau salin `buildozer.spec` ke folder proyek.


In [ ]:
%cd "$PROJECT_DIR"
!echo '=== folder content ==='
!ls -la
!echo '=== check buildozer.spec ==='
!if [ -f buildozer.spec ]; then echo 'buildozer.spec found'; else echo 'ERROR: buildozer.spec missing'; fi


In [ ]:
%cd "$PROJECT_DIR"
!buildozer android debug


## 7. Simpan APK ke Drive
Setelah Buildozer selesai, APK akan berada di `bin/`. Salin ke Drive untuk diambil.


In [ ]:
!echo 'APK ready at:'
!ls -la "$PROJECT_DIR"/bin/*.apk


## Catatan Penting
- Buildozer akan mengunduh SDK/NDK dan komponen Android, prosesnya bisa memakan waktu lebih dari 30 menit.
- Jika terjadi error `No such file or directory` atau `BUILD FAILED`, periksa versi `buildozer.spec`, API level, dan dependensi Android.
- Colab bisa kehabisan ruang jika paket terlalu berat; pakai `!df -h` untuk cek sisa disk.
